# CVRPTW Project Notebook

This notebook combines the project data structures, pseudocode notes, and the current `main.py` content.


In [ ]:
class Delivery:
    def __init__(self, id, demand, start, end, location):
        self.DeliveryID = id
        self.Demand = demand
        self.TimeStart = start
        self.TimeEnd = end
        self.Location = location

# In a vehicle routing problem, you usually need to track many deliveries at once.
# Instead of storing them as loose variables, I have stored each one as an object
# with all relevant attributes. That makes the algorithm easier to write and reason about.

class Vehicle:
    def __init__(self, id, capc, depot, start, end):
        self.VehicleID = id
        self.Capacity = capc
        self.DepotLocation = depot
        self.ShiftStart = start
        self.ShiftEnd = end

class Route:
    def __init__(self, vehicle):
        self.Vehicle = vehicle
        self.Stops = []
        self.TotalLoad = 0
        self.TotalDistance = 0.0
        self.TotalDuration = 0.0

    def add_stop(self, delivery):
        self.Stops.append(delivery)
        self.TotalLoad += delivery.Demand

    def set_totals(self, distance, duration):
        self.TotalDistance = distance
        self.TotalDuration = duration

    def __len__(self):
        return len(self.Stops)

    def __repr__(self):
        return (
            f"Route(VehicleID={self.Vehicle.VehicleID}, "
            f"Stops={len(self.Stops)}, TotalLoad={self.TotalLoad}, "
            f"TotalDistance={self.TotalDistance}, TotalDuration={self.TotalDuration})"
        )

Example_delivery1 = Delivery(1, 5, 9, 11, (51.5074, -0.1278))
Example_vehicle1 = Vehicle(1, 100, (51.5000, -0.1200), 8, 18)


## Pseudocode for the Development of the Solution
Design-section pseudocode, feeding directly into Technical Solution later.

### Data Structures
Here is an example pseudocode that showcases the possible data structures I will be using in my technical solution. These abstract data structures map directly onto the ones shown at section 3.3.2 [here](../main.pdf#page=25).

```py
class Delivery:
    #stores all relevant information for deliveries
    id, demand, time_window_start, time_window_end, 
    service_duration, location #(lat, lon)

class Vehicle:
    #stores all relevant information for vehicles
    id, capacity, depot_location, 
    shift_start, shift_end, max_route_duration #(optional)

class Route:
    vehicle, total_load, total_distance, 
    total_duration, stops #(ordered list of Delivery)
```

The use of classes here helps me
- Keep my code DRY (Don't Repeat Yourself).
- Makes code easier to maintain, reuse, and debug.
- Provides a clear structure to my solution.
- Allows me to build reusable structures to my code

OOP is an example of object-oriented decomposition. While functional or procedural paradigms break a system down by what it does (actions and functions), OOP breaks a system down by what things exist (entities and objects). This should help to critically reduce chaotic dependencies across the entire codebase.

---

### Phase 1: Construct Routes
this can be seen here (process 4 at [here](../main.pdf#page=26) at page 26.)

```py
function construct_routes(deliveries, vehicles, distance_matrix, depot):
    unallocated = copy(deliveries)
    routes = []

    for vehicle in vehicles:
        if unallocated is empty: break
        route = new Route(vehicle)
        location, time, load = depot, vehicle.shift_start, 0

        while True:
            candidates = [d for d in unallocated if
                load + d.demand <= vehicle.capacity
                and time + travel_time(location, d.location) <= d.time_window_end]

            if candidates is empty: break

            next_stop = closest(candidates, location, distance_matrix)
            arrival = time + travel_time(location, next_stop.location)
            start_service = max(arrival, next_stop.time_window_start)

            route.add(next_stop, start_service)
            time = start_service + next_stop.service_duration
            load += next_stop.demand
            location = next_stop.location
            unallocated.remove(next_stop)

        routes.append(route)

    return routes, unallocated
```

A delivery that would break capacity or arrive too late is simply never in `candidates` — it can't get added, so there's nothing to reject later.

---

### Phase 2 — Check constraints
This selection will aim to match process 5 in [here](../main.pdf#page=26) at page 26.

Now this only has to check things that genuinely can't be known per stop during construction:

```py
function check_constraints(routes, unallocated):
    violations = []
    if unallocated is not empty:
        violations.append("Unallocated deliveries", unallocated)
    for route in routes:
        if route.total_duration > route.vehicle.max_route_duration:
            violations.append(route, "Shift limit exceeded")
    return violations
```

Per-stop capacity/time-window feasibility never appears here — it can't be violated, by construction.

### Phase 3 — Improve routes
Plain 2-opt only checks distance, but your problem has time windows. Reversing a segment can shorten total distance while pushing a later stop's arrival past its window — reordering changes arrival times, even though it doesn't change total load. So every candidate swap needs a feasibility re-check and not just a distance comparison.

```py
function improve_route(route, distance_matrix):
    improved = true
    while improved:
        improved = false
        for i, j in segment_pairs(route.stops):
            candidate = reverse_segment(route.stops, i, j)
            if distance(candidate) < distance(route.stops)
               and time_windows_feasible(candidate, route.vehicle):
                route.stops = candidate
                improved = true
    return route
```

Capacity doesn't need re-checking here (reordering doesn't change total load as $\alpha + \beta = \beta + \alpha$), but time windows do.

---

## CVRPTW System — Function IPO Diagrams (D2)
One diagram per process from the system DFD. Each is a self-contained D2 file — inputs enter the highlighted box from the left, outputs leave from the right (`direction: right` drives this layout). Plain-text nodes (no border) represent the external entity, process, or data store on the other end of each flow — they exist to label where data comes from / goes to, not as part of the function itself.

**To render:** paste any block into https://play.d2lang.com, or run `d2 diagram.d2 diagram.svg` locally, then `\includegraphics` the SVG into your LaTeX report.

Dashed edges (`style.stroke-dash`) mark feedback/exception flows.

---

## 1. Validate orders and vehicles

```d2
direction: right

Delivery_coordinator: "Delivery\ncoordinator" { shape: text }

fn: "1. Validate orders\nand vehicles" {
  shape: rectangle
  style.fill: "#4C8BF5"
  style.font-color: white
}

Geocode: "Geocode and\nconfirm address" { shape: text }
Vehicle_store: "Driver and\nvehicle records" { shape: text }

Delivery_coordinator -> fn: "Orders, vehicles\nand depot"
fn -> Geocode: "Validated orders &\nvehicle records"
fn -> Vehicle_store: "Vehicle records"
```
